# 08 - Agents

Demonstrate AIMU's autonomous agents: an LLM that loops over its own tool calls until it decides to stop. The agent owns the control flow — you give it tools and a goal; the model picks which tools to call, in what order, and when to finish.

This notebook covers:

- `Agent` — the basic agentic loop with `@tool`-decorated functions and MCP tools
- System messages and per-config construction
- Streaming with `StreamChunk` (`phase`, `content`, `agent`, `iteration`)
- `agent.as_model_client()` — drop an agent into any API that wants a `BaseModelClient`

For code-controlled patterns (`Chain`, `Router`, `Parallel`, `EvaluatorOptimizer`, `PlanExecuteEvaluator`) see **notebook 10**. For filesystem-discovered skills (`SkillAgent`) see **notebook 09**. For ready-to-use orchestrator agents see **notebook 11**.

## Quick start: `@tool` and `Agent`

The shortest path to a working agent: decorate any Python function with `@tool` and pass it to an `Agent`. The decorator inspects the signature and docstring to build an OpenAI-format tool spec; the agent registers the function on its `ModelClient` and dispatches calls in-process.

Use this route for tools that live in the same process as your code. For cross-process tools (a shared tool server, a fleet of agents sharing a catalog), use `MCPClient` as shown in the next section. The two routes can be combined on the same agent.

In [ ]:
from aimu.models import ModelClient, OllamaModel
from aimu.agents import Agent
from aimu.tools import tool


@tool
def letter_counter(word: str, letter: str) -> int:
    """Count occurrences of a letter in a word."""
    return word.lower().count(letter.lower())


quickstart_client = ModelClient(OllamaModel.QWEN_3_5_9B)
quickstart_agent = Agent(quickstart_client, tools=[letter_counter], max_iterations=3)

print(quickstart_agent.run("How many r's are in the word strawberry?"))

## Setup

Create an `OllamaClient` with a tool-capable model and attach an `MCPClient` with a few demo tools. The same setup is used across all sections.

In [ ]:
import datetime
from fastmcp import FastMCP

from aimu.models import OllamaClient
from aimu.tools import MCPClient

mcp = FastMCP("Demo Tools")


@mcp.tool()
def get_current_date_and_time() -> str:
    """Returns the current date and time."""
    return str(datetime.datetime.now())


@mcp.tool()
def get_weather(location: str) -> str:
    """Returns the current weather for a given location."""
    # Stubbed for demo purposes
    return f"Sunny, 22Â°C in {location}"


@mcp.tool()
def calculate(expression: str) -> str:
    """Evaluates a simple arithmetic expression and returns the result."""
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"Error: {e}"


model_client = OllamaClient(OllamaClient.MODELS.QWEN_3_5_9B)
model_client.mcp_client = MCPClient(server=mcp)

print("Model:", model_client.model.value)
print("Tools:", [t.name for t in model_client.mcp_client.list_tools()])

## A - Basic Agent

A `Agent` wraps a `ModelClient` and runs an agentic loop: after each `chat()` call, if the model used tools, the agent sends a continuation prompt to allow further tool use. The loop stops once the model responds without calling any tools.

This is the simplest possible usage â€” two lines.

In [ ]:
from aimu.agents import Agent

model_client.messages = []

agent = Agent(model_client, name="assistant", max_iterations=5)
result = agent.run("Get the current date and time. Subsequently calculate 123 * 456.")

print(result)

The agent called both tools â€” date/time and calculator â€” and assembled the results into a final answer. Inspect the full message history to see every step.

In [ ]:
model_client.messages

## B - Agent with System Message

Pass a `system_message` to give the agent a specific persona or set of instructions.

In [ ]:
model_client.messages = []

agent = Agent(
    model_client,
    name="weather-bot",
    max_iterations=5,
)
model_client.system_message = "You are a travel assistant. Always check the weather before giving advice. Be concise. If you've answered the original query and don't need to use any more tools, repeat the final answer."

result = agent.run("Should I visit Paris or London today?")
print(result)

In [ ]:
model_client.messages

## C - Agent from Config

`Agent.from_config()` creates an agent from a plain dict. This makes it easy to define agents in configuration files or environment settings without writing boilerplate code.

In [ ]:
agent_config = {
    "name": "calculator-agent",
    "system_message": "You are a maths assistant. Use the calculate tool for all arithmetic. If you've answered the original query and don't need to use any more tools, repeat the final answer.",
    "max_iterations": 8,
    "continuation_prompt": "Continue solving the problem step by step.",
}

model_client.messages = []
agent = Agent.from_config(agent_config, model_client)

print(f"Agent name:       {agent.name}")
print(f"Max iterations:   {agent.max_iterations}")
print(f"System message:   {model_client.system_message}")

In [ ]:
result = agent.run("What is (17 * 23) + (88 / 4)?")
print(result)

In [ ]:
model_client.messages

## D - Streaming Agent Output

`run(stream=True)` yields `StreamChunk` objects — each with `phase`, `content`, `agent`, and `iteration`. The same chunk type flows through `client.chat(stream=True)` and every workflow; `agent` and `iteration` are populated by the agent/workflow runner (and default to `None` / `0` for plain client chats).

The `iteration` field shows which loop round produced each chunk, making it easy to follow multi-step reasoning. Use `chunk.is_text()` / `chunk.is_tool_call()` to dispatch without writing out the phase comparison.

In [ ]:
from aimu.models import StreamingContentType

model_client.messages = []
model_client.system_message = None

agent = Agent(model_client, name="streamer", max_iterations=6)

current_iteration = -1
current_phase = None

for chunk in agent.run("What is the weather in Tokyo, and what is 99 * 11?", stream=True):
    if chunk.iteration != current_iteration:
        current_iteration = chunk.iteration
        print(f"\n--- iteration {current_iteration} ---")

    if chunk.phase != current_phase:
        current_phase = chunk.phase
        print(f"\n--- phase {current_phase.value} ---")

    if chunk.phase == StreamingContentType.THINKING:
        print(f"{chunk.content}", end="", flush=True)
    elif chunk.phase == StreamingContentType.TOOL_CALLING:
        print(f"{chunk.content['name']} > {chunk.content['response']}")
    elif chunk.phase == StreamingContentType.GENERATING:
        print(chunk.content, end="", flush=True)

In [ ]:
model_client.messages

## E - agent.as_model_client()

`agent.as_model_client()` returns a `BaseModelClient` view backed by the agent. Use it anywhere a plain model client is accepted (web UIs, conversation managers, etc.) to get the full agentic loop transparently.

`chat()` on the returned view runs the complete `Agent` loop and returns once the model stops calling tools. `generate()` bypasses the loop and calls the inner client directly. State (`messages`, `system_message`, `mcp_client`) delegates to `agent.model_client`, so the view is a true drop-in replacement.

To run a workflow (`Chain`, `Router`, etc.), call its `run()` or `run(stream=True)` directly; see **notebook 10** for workflow patterns.

In [ ]:
client = OllamaClient(OllamaClient.MODELS.QWEN_3_5_9B)
client.mcp_client = MCPClient(server=mcp)

agentic_client = Agent(client, max_iterations=5).as_model_client()

# chat() drives the full agentic loop — identical call site to a plain ModelClient
result = agentic_client.chat("What is the weather in Berlin, and what is 42 * 7?")
print(result)

## F - Clean Up

Delete model clients to release any held resources.

In [ ]:
del model_client, agentic_client